[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataguirre/Curso-IA-Aplicada/blob/main/Semana%2009-10_%20NLP_y_etica/fundamentos_nlp.ipynb)



# Fundamentos de NLP

In [ ]:
!pip install pandarallel
!pip install gensim

In [ ]:
import re
import nltk
import pandas as pd
from tqdm import tqdm
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# --- Descargas necesarias ---
nltk.download("stopwords")

# Base de datos Banco de la Republica

In [ ]:
banrep = pd.read_csv('https://drive.google.com/uc?id=1vz3530nztdYPs-x0ZELlRPjNEcK453hc')
banrep

In [ ]:
columns = ['uuid', 'name', 'collection', 'authors', 'type', 'date', 'abstract_spa',
           'doi_link', 'access_rights', 'access_rights', 'keywords_spa', 'jel_spa']

banrep = banrep[(~banrep['abstract_spa'].isna()) & (~banrep['authors'].isna())][columns].reset_index(drop=True)
banrep

# Preprocesamiento de texto

Expresiones Regulares

Las *expresiones regulares* también conocidas como *regex* o *regexp* son secuencias de caracteres que conforman patrones de búsqueda y tienen la ventaja de ser transversales a los distintos lenguajes de programación.

Para ilustrar a qué nos referimos, comencemos con un ejemplo sencillo. El patrón más sencillo que se puede utilizar con expresiones regulares es utilizar secuencia de caracteres que uno quiere encontrar en el texto. Por ejemplo, si quisiéramos buscar la palabra *tienda* en un texto, simplemente podríamos usar como patrón `tienda`. Los patrones de búsqueda pueden estar conformados por un solo caracter como `!` para buscar signos de exclamación o también una secuencia de letras:

<div> <center>

| **RE** |      **Ejemplo del patrón capturado**     |
|:------:|:-----------------------------------------:|
| tienda | El que tenga <u>tienda</u> que la atienda |
|    a   |    El que m<u>a</u>druga Dios le ayuda    |
|    !   |            ¡Ojo con eso<u>!</u>           |

</center> </div>   

Este tipo de búsquedas es sensible al uso de mayúsculas, por ejemplo, buscar la palabra `tienda` arroja un resultado diferente al de buscar `Tienda`. Del mismo modo, también es sensible al uso de caracteres especiales como tildes, apostrofes, etc. En la práctica se suelen eliminar estos caracteres especiales para simplificar el texto analizado. Por ejemplo, transformar un texto como:

**<center> A palabras necias oídos sordos </center>**

por

**<center> a palabras necias oidos sordos </center>**

hará más sencillo su tratamiento. No obstante, las expresiones regulares son una herramienta muy poderosa y nos permiten usar funciones que simplifican la tarea. Por ejemplo, podemos usar los corchetes (`[]`) para expresar disyunción lógica (`o`). Por ejemplo, la búsqueda `[Tt]ienda` sirve para encontrar la palabra `tienda` **o** la palabra `Tienda`. Los corchetes indican que se busca una palabra que contenga la cadena `ienda` precedida por una letra `t` en minúscula **o** mayúscula. Por ejemplo:

<div> <center>

|    **RE**    |**Patrón capturado**|          **Ejemplo del patrón capturado**            |
|:------------:|:----------------:|:------------------------------------------------------:|
|   [Tt]ienda  |  Tienda o tienda |        El que tiene <u>tienda</u> que la atienda       |
|     [abc]    |   a, b **o** c   | No me <u>a</u>bra los ojos que no le voy a echar gotas |
| [1234567890] | Cualquier dígito |            Eramos entre <u>5</u> y 8 personas   |

</div> </center>

Notemos en la última linea que la expresión regular `[1234567890]` nos permite capturar cualquier dígito, no obstante, escribir bloques de dígitos o letras puede ser inconveniente. Es decir, para capturar cualquier letra no es práctico escribir todo el abecedario: `[abcdefghijklmnopqrstuvwxyz]`. En estos casos uno puede completar la búsqueda dentro de corchetes con un guion (`-`) que especifica rangos. Por ejemplo `[0-9]` nos permite capturar cualquier número entre 0 y 9, `[b-g]` nos permite capturar cualquier letra de la `b` a la `g` o sea *b, c, d, e, f **o** g*.

<div> <center>

| **RE** |     **Patrón capturado**     |            **Ejemplo del patrón capturado**            |
|:------:|:----------------------------:|:------------------------------------------------------:|
|  [A-Z] | Cualquier letra en mayúscula |        <u>E</u>l que tiene tienda que la atienda       |
|  [a-z] | Cualquier letra en minúscula | N<u>o</u> me abra los ojos que no le voy a echar gotas |
|  [0-9] |       Cualquier dígito       |       Eramos entre <u>5</u> y 8 personas       |

</div> </center>

Podemos también indicar que caracteres no deben ser capturados, para ello utilizamos un caret (`^`) al inicio del corchete `[^]`. Sólo si el caret (`^`) es el primer símbolo dentro del corchete, el patrón subsiguiente es negado. Por ejemplo, `[^a]` significa que se va a capturar cualquier caracter, incluyendo los especiales, excepto la letra *a*.

<div> <center>

| **RE** |               **Patrón capturado**              |            **Ejemplo del patrón capturado**            |
|:------:|:-----------------------------------------------:|:------------------------------------------------------:|
| [^A-Z] | Cualquier caracter menos una letra en mayúscula |       E<u>l</u> que tiene tienda que la atienda       |
|  [^Ss] |     Cualquier caracter excepto una "s" o "S"    | <u>N</u>o me abra los ojos que no le voy a echar gotas |
|  [^.]  |        Cualquier caracter menos un punto        |       <u>E</u>ramos al rededor de 5 a 8 personas       |
|  [e^]  |             Captura una "e" o un "^"            |                       <u>e</u>^x                       |
|  [a^b] |                   Captura a^b                   |                 La expresión <u>a^b</u>                |

</div> </center>

Note sin embargo, que si se usa el caret (`^`) en cualquier otro lugar de la expresión regular, este no va a significar una negación, sino un caret.

Del mismo modo, a menudo buscamos capturar patrones opcionales. Por ejemplo, para capturar una palabra en plural o en singular en donde el último caracter es una *s*. Para esto utilizamos el símbolo de pregunta (`?`) después del caracter opcional. El signo de pregunta (`?`) en el contexto de expresiones regulares significa el caracter anterior o ninguno.


<div> <center>

|  **RE**  | **Patrón capturado** |            **Ejemplo del patrón capturado**           |
|:--------:|:--------------------:|:-----------------------------------------------------:|
| tiendas? | "tienda" o "tiendas" |       El que tenga <u>tienda</u> que la atienda       |
|  colou?r |  "color" o "colour"  | Discover the newest hand-picked <u>color</u> palettes |

</div> </center>

Pero también existen casos donde un caracter se puede repetir más de una vez. Por ejemplo, en un libro se podría encontrar la onomatopeya del mujido de una vaca de diversas formas:

**<center> Muu! </center>**

**<center> Muuu! </center>**

**<center> Muuuu! </center>**

**<center> Muuuuu! </center>**

A grandes rasgos, podemos describir esta onomatopeya como una palabra que comienza con una *M* seguida con por lo menos dos *u* y finaliza con el signo de exclamación *!*. La expresión regular que nos permite capturar cero o más ocurrencias de un caracter es el asterisco (`*`) también conocido como *cleany star* o *Kleene* . Por ende, la expresión regular `u*` va a capturar tanto `u` como `uuuuuu`, pero a su vez también podría capturar `vaca` pues esta palabra tiene cero letras u.

Para corregir esto, podríamos usar la expresión regular `uu*` la cual significa una o más letras u. Algunos patrones más complejos también se pueden utilizar haciendo uso de los corchetes; por ejemplo `[ab]\*` sirve para capturar cero o más *a*s o *b*s. Por ende, se capturarían textos como *aaaaa*, *bbb* o *ababababab*.

Asimismo, para especificar múltiples dígitos podemos usar `[0-9][0-9]*` para capturar cualquier entero.

Sin embargo, aún podemos utilizar el signo de suma (`+`), también llamado *Kleene +*, para simplificar las expresiones regulares. El *Kleene +*, nos permite denotar que el caracter a capturar se repite una o más veces. Por ende, la expresión `[0-9]+` es la forma más común de expresar una secuencia de dígitos. Por ejemplo:

<div> <center>

|  **RE**  | **Patrón capturado** |            **Ejemplo del patrón capturado**           |
|:--------:|:--------------------:|:-----------------------------------------------------:|
|   baa*   |  ba con una o más as |               La cabra hace <u>baaa</u>!              |
|   mu+!   | mu! con una o más us |              La vaca hizo <u>muuuuu!</u>          |
|  [0-9]+  |   Cualquier entero   |              Ese camisa cuesta $<u>25</u>             |

</div> </center>

Otra función importante esta dada por el punto (`.`). Este funciona como comodín o *wildcard*. Esta expresión regular sirve para capturar cualquier caracter excepto los saltos de línea.

<div> <center>

|  **RE**  | **Patrón capturado** |            **Ejemplo del patrón capturado**           |
|:--------:|:--------------------:|:-----------------------------------------------------:|
   |
|   1. | 10 y 1A             | Ganaron el partido <u>18</u> a 2             |
|  1.4 | 114 y 1B4            | Vive en el apartamento <u>1B4<u> |

</div> </center>

También existen los denominado anclas o *anchors* que sirven para capturar elementos en posiciones particulares del texto. Los más comunes son el caret (`^`) y el símbolo de dólar (`$`) los cuales hacen alusión al inicio y final de un texto respectivamente. Por ejemplo, la expresión `^El` solo captura la palabra *El* sólo si está al inicio del corpus de texto. Otras anclas comunes son (`\b`) y (`\B`) que denotan los *boundaries* o límites de una palabra o dentro de una palabra respectivamente. Por ejemplo, `\bel\b` va a capturar la palabra *el* pero no *elefante*.

### Algunos operadores

<div> <center>

| **RE** | **Expansión** |                          **Patrón capturado**                         |
|:------:|:-------------:|:---------------------------------------------------------------------:|
|   \d   |     [0-9]     |                            Cualquier dígito                           |
|   \D   |     [^0-9]    |                          Cualquier no dígito                          |
|   \w   |  [a-zA-Z0-9_] |                  Cualquier alfanumérico o guion bajo                  |
|   \W   |     [ˆ\w]     |                 Cualquier no alfanumérico o guion bajo                |
|   \s   |  [ \r\t\n\f]  |                           Espacio en blanco                           |
|   \S   |     [ˆ\s]     |                        No un espacio en blanco                        |
|    *   |               |         Cero o más ocurrencias del caracter o expresión pasada        |
|    +   |               |         Una o más ocurrencias del caracter o expresión pasada         |
|    ?   |               | Exactamente cero o una ocurrencia del del caracter o expresión pasada |
|   {n}  |               |            *n* ocurrencias del caracter o expresión pasada            |
|  {n,m} |               |        De *n* a *m* ocurrencias del caracter o expresión pasada       |
|  {n,}  |               |      Por lo menos *n* ocurrencias del caracter o expresión pasada     |
|  {,m}  |               |         Hasta *m* ocurrencias del caracter o expresión pasada         |

</div> </center>

In [ ]:
# --- Recursos globales ---
STOPWORDS_ES = set(stopwords.words("spanish")) | {
    "doi", "http", "https", "www", "et", "al", "etc"
}
STEMMER_ES = SnowballStemmer("spanish")

# Mapa para quitar acentos (manteniendo ñ)
_ACCENT_MAP = str.maketrans("áéíóúüÁÉÍÓÚÜ", "aeiouuAEIOUU")

# Expresiones regulares útiles

# Detecta URLs
_URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)

# Detecta correos electronicos
_EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")

# Detecta números o símbolos comunes no textuales
_NUM_SYM = re.compile(r"[0-9]+|[_/#%€$£°ºª©™®§]")

# Detecta secuencias de letras (tokens) en minúscula
_TOKEN_RE = re.compile(r"[a-zñ]+")

In [ ]:
def normalizacion(texto):
    """
    Normaliza texto en español:
    - Minúsculas
    - Quita URLs, emails, números y símbolos
    - Elimina tildes y diéresis (mantiene ñ)
    - Colapsa espacios múltiples
    """
    if not isinstance(texto, str):
        return ""

    t = texto.strip().lower()
    t = _URL_RE.sub(" ", t)
    t = _EMAIL_RE.sub(" ", t)
    t = _NUM_SYM.sub(" ", t)
    t = t.translate(_ACCENT_MAP)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def tokenizar_texto(texto):
    """
    Tokeniza extrayendo palabras (incluye ñ).
    """
    t = normalizacion(texto)
    return _TOKEN_RE.findall(t)

def eliminar_stopwords(tokens):
    """
    Elimina stopwords en español.
    """
    return [w for w in tokens if w not in STOPWORDS_ES and len(w) > 1]

def stemmizar(tokens):
    """
    Aplica stemming en español usando SnowballStemmer.
    """
    return [STEMMER_ES.stem(w) for w in tokens]

def preprocesamiento(texto):
    """
    Pipeline completo: normaliza → tokeniza → quita stopwords → stemming.
    Devuelve lista de tokens procesados.
    """
    tokens = tokenizar_texto(texto)
    tokens = eliminar_stopwords(tokens)
    tokens = stemmizar(tokens)
    return tokens

In [ ]:
ejemplo = banrep['abstract_spa'].iloc[0]
print("Texto original: \n")
print(ejemplo)

In [ ]:
# Normalización
ejemplo_norm = normalizacion(ejemplo)
print("Texto normalizado:")
print(ejemplo_norm, "\n")

In [ ]:
# Tokenización
ejemplo_tokens = tokenizar_texto(ejemplo)
print("Tokens:")
print(' '.join(ejemplo_tokens), "\n")

In [ ]:
# Eliminación de stopwords
ejemplo_nosw = eliminar_stopwords(ejemplo_tokens)
print("Sin stopwords:")
print(' '.join(ejemplo_nosw), "\n")

In [ ]:
# Stemming
ejemplo_stem = stemmizar(ejemplo_nosw)
print("Con stemming:")
print(' '.join(ejemplo_stem))

In [ ]:
tqdm.pandas(desc="Preprocesando abstracts ES")
banrep["tokens_clean"] = (
    banrep["abstract_spa"]
    .fillna("")
    .astype(str)
    .progress_apply(preprocesamiento)
)

**Paralelizacion?**

La **paralelización** consiste en dividir un conjunto de tareas en varios procesos que se ejecutan **simultáneamente** en distintos núcleos del procesador (CPUs).  
En lugar de procesar cada texto de forma secuencial con un solo hilo, el sistema reparte el trabajo entre varios *workers* que operan en paralelo, reduciendo significativamente el tiempo total de ejecución.

En tareas como el **preprocesamiento de texto**, esta técnica resulta especialmente eficaz porque cada observación (por ejemplo, cada *abstract*) se puede limpiar o tokenizar **de manera independiente** del resto.  
Esa independencia permite distribuir las operaciones entre múltiples núcleos sin riesgo de conflictos o dependencia entre procesos.

En este ejemplo, la librería `pandarallel` divide el `DataFrame` en fragmentos, los envía a los *workers* disponibles y combina los resultados al finalizar, mostrando además una barra de progreso.  
Gracias a esto, el preprocesamiento se completa **en una fracción del tiempo** comparado con el uso tradicional de `apply` o `progress_apply`.

En resumen:  
- Aprovecha todos los núcleos disponibles del CPU.  
- Reduce tiempos en tareas repetitivas e independientes.  
- Es ideal para pipelines de limpieza o tokenización de texto.

In [ ]:
from pandarallel import pandarallel
import os
import time

print(f'CPUs: {os.cpu_count()}')

In [ ]:
workers = 2
pandarallel.initialize(progress_bar=True, nb_workers=workers)

start_time = time.time()

banrep["tokens_clean"] = (
    banrep["abstract_spa"]
    .fillna("")
    .astype(str)
    .parallel_apply(preprocesamiento)
)

end_time = time.time()
elapsed = end_time - start_time

print(f"Preprocesamiento completado en {elapsed:.2f} segundos")

# Nube de Palabras

Con nuestro texto normalizado, podemos hacer un análisis sencillo del texto usando nubes de palabras. Las nubes de palabras (también conocidas como "word clouds" en inglés) son representaciones visuales de un conjunto de palabras en un texto, donde el tamaño de cada palabra se determina en función de su frecuencia de aparición en el texto. Es una forma popular y efectiva de visualizar la distribución y relevancia de las palabras en un documento.

La librería `WordCloud` en `Python` es una herramienta ampliamente utilizada para crear nubes de palabras de manera sencilla. La función principal de `WordCloud` es tomar un texto y generar una nube de palabras donde el tamaño de cada palabra se determina por su frecuencia en el texto.

In [ ]:
tqdm.pandas(desc="Preprocesando abstracts ES")
banrep["tokenizar"] = (
    banrep["abstract_spa"]
    .fillna("")
    .astype(str)
    .progress_apply(tokenizar_texto)
)
docs_tokens = banrep['tokenizar'].dropna().tolist()
text = " ".join(" ".join(doc) for doc in docs_tokens)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS
from gensim.models.phrases import Phrases, Phraser
from tqdm import tqdm

# Crear y mostrar la nube
wc = WordCloud(
    width=1600, height=800,
    background_color="white",
    stopwords=STOPWORDS_ES,
    collocations=False  # evita juntar palabras comunes que no quieres
).generate(text)

plt.figure(figsize=(18,9))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()

In [ ]:
docs = banrep["tokenizar"].dropna().tolist()

# ------------------------------------------------------------
# Detectar bigramas frecuentes con Gensim
# ------------------------------------------------------------
# min_count: número mínimo de veces que un par debe aparecer
# threshold: entre más alto, menos bigramas (ajusta según corpus)
bigram_model = Phrases(docs, min_count=10, threshold=20)
bigram_phraser = Phraser(bigram_model)

# Aplicar el modelo a todos los documentos
docs_with_bigrams = [bigram_phraser[doc] for doc in tqdm(docs, desc="Detectando bigramas")]

# ------------------------------------------------------------
# Aplanar todo el corpus a una sola lista de tokens
# ------------------------------------------------------------
tokens_flat = [token for doc in docs_with_bigrams for token in doc]

# Opcional: eliminar tokens muy cortos o numéricos
tokens_flat = [t for t in tokens_flat if len(t) > 2 and not t.isnumeric()]

# ------------------------------------------------------------
# Generar el texto completo para la nube
# ------------------------------------------------------------
text = " ".join(tokens_flat)

wc = WordCloud(
    width=1600,
    height=800,
    background_color="white",
    stopwords=STOPWORDS_ES,
    collocations=False,   # evita duplicar bigramas automáticos
    max_words=200
).generate(text)

plt.figure(figsize=(18,9))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.show()

# TF-IDF

$$
tf-idf_{ij}=tf_{ij} \times \left( \log \left( \frac{1+N}{1+df_i} \right)+1\right)
$$

donde:

- $tf_{ij}$ es la frecuencia palabra $i$ en el documento $j$
- $df_{ij}$ es el número de documentos que contienen la palabra $i$
- $N$ es el número de documentos

# Sckit-Learn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.85,
    sublinear_tf=True,
    dtype=np.float32
)

X_tfidf = vectorizer.fit_transform(banrep["abstract_spa"].fillna("").astype(str))
feat_names = vectorizer.get_feature_names_out()
print("Dimensión TF-IDF:", X_tfidf.shape)

In [ ]:
feat_names[1000:1010]

In [ ]:
X_tfidf

In [ ]:
# Como buscamos en un sparse?
row = X_tfidf[10]

# índices de columnas con valores diferentes a 0
idxs = row.indices

# valores TF-IDF asociados a esas columnas
vals = row.data

for i, v in zip(idxs[:10], vals[:10]):
    print(f"{feat_names[i]:25s} {v:.4f}")

# Gensim

In [ ]:
from gensim.corpora import Dictionary
from gensim.models import TfidfModel
from gensim import matutils
import numpy as np

# Hacemos nuestro corpus de documentos en una lista
texts = banrep["tokens_clean"].tolist()

# Creamos un diccionario de palabras unicas
dictionary = Dictionary(texts)

# Ajusta estos umbrales a tu tamaño de corpus
dictionary.filter_extremes(no_below=5, no_above=0.85)  # >=5 docs, <=85% de docs
dictionary.compactify()

# BoW (lista de listas de (term_id, count))
corpus_bow = [dictionary.doc2bow(doc) for doc in texts]

print(dictionary.token2id)

In [ ]:
# TF-IDF con Gensim

# Calcula los df y idf
tfidf_model = TfidfModel(corpus_bow, smartirs='ntc', normalize=True)

# Ya calcula la matriz TF-IDF
corpus_tfidf = tfidf_model[corpus_bow]  # iterable de listas (term_id, tfidf)

corpus_tfidf

In [ ]:
# Buscar palabras mas frecuentes del documento 0
doc_tfidf = corpus_tfidf[0]

# Ordenar por peso TF-IDF descendente
doc_tfidf_sorted = sorted(doc_tfidf, key=lambda x: x[1], reverse=True)

# Mostrar los top 10 términos
top_k = 10
top_terms = [(dictionary[id], float(score)) for id, score in doc_tfidf_sorted[:top_k]]
print(f"Top {top_k} términos del documento 0:\n", top_terms)

# Distancia del coseno

In [ ]:
from gensim.similarities import MatrixSimilarity
import numpy as np

# Crear índice de similitud TF-IDF
index = MatrixSimilarity(corpus_tfidf, num_features=len(dictionary))

# Definir función de búsqueda
def buscar_documentos(query, top_n=5):
    """
    Busca en el corpus los documentos más similares a la query.
    Imprime título, similitud y fragmento del abstract.
    """
    # Preprocesar la consulta (igual que los textos del corpus)
    tokens = preprocesamiento(query)
    bow = dictionary.doc2bow(tokens)
    tfidf_vec = tfidf_model[bow]

    # Calcular similitud coseno entre query y documentos
    sims = index[tfidf_vec]  # vector de similitudes

    # Top-N documentos más similares
    top_idx = np.argsort(sims)[::-1][:top_n]

    for i in top_idx:
        print(f"{banrep.loc[i, 'name']}")
        print(f"   Similitud: {float(sims[i]):.4f}")
        print(f"   Extracto: {banrep.loc[i, 'abstract_spa'][:600].strip()}\n")
        print(f"   UUID: {banrep.loc[i, 'uuid']}\n")
        print("-" * 100)
    return None

Revisemos diferentes consultas a nuestra matriz TF-IDF y comparemos con el buscador del repositorio del banco de la republica ([link](https://repositorio.banrep.gov.co/search?spc.page=1))

In [ ]:
# impacto de la política monetaria en la inflación colombiana
# evaluacion de impacto
# economía de Suroccidente
# balanza de pagos
# evolucion del endeudamiento externo
# consumo de los hogares

from ipywidgets import interact, Text, IntSlider

interact(
    buscar_documentos,
    query=Text(value="regla fiscal", description="Query:"),
    top_n=IntSlider(value=5, min=1, max=15, step=1, description="Top-N")
)